In [42]:
import ast
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

try:
    import wandb
except ImportError:
    wandb = None

BLOCK_SIZE = 64
ANNOTATIONS_PATH = "../data/annotations.csv"
EMBEDDINGS_DIR = Path("../embeddings/train")
WANDB_PROJECT = "action-classifier"
USE_WANDB = wandb is not None

VERB_CONFIG = {
    "block_size": BLOCK_SIZE,
    "epochs": 30,
    "batch_size": 256,
    "lr": 2e-4,
    "weight_decay": 3e-4,
    "dropout": 0.25,
    "hidden_dims": [1536,768],
    "test_size": 0.2,
    "random_state": 42,
}

NOUN_CONFIG = {
    "block_size": BLOCK_SIZE,
    "epochs": 30,
    "batch_size": 128,
    "lr": 2e-4,
    "weight_decay": 3e-4,
    "dropout": 0.4,
    "hidden_dims": [1536,768],
    "test_size": 0.2,
    "random_state": 42,
}

In [43]:
# Load only relevant annotations
df = pd.read_csv(ANNOTATIONS_PATH)
df = df[df["relevant"] == True].copy()

# Convert frame ranges to block ranges
df["start_block"] = df["start_frame"] // BLOCK_SIZE
df["stop_block"] = df["stop_frame"] // BLOCK_SIZE

# Expand each action to one row per block it spans
rows = []
for _, row in df.iterrows():
    # all_noun_classes is a list like [3] or [1, 2, 5] — an action can have multiple nouns.
    # We use the first one as the primary noun class, which corresponds to the primary noun.
    # If a block has multiple overlapping actions, we later resolve the conflict by majority vote.
    primary_noun_class = ast.literal_eval(row["all_noun_classes"])[0]
    for block in range(row["start_block"], row["stop_block"] + 1):
        rows.append({
            "video_id": row["video_id"],
            "block": block,
            "verb_class": row["verb_class"],
            "noun_class": primary_noun_class,
        })

expanded = pd.DataFrame(rows)

# If multiple actions overlap the same block, pick the most frequent label
verb_labels = expanded.groupby(["video_id", "block"])["verb_class"].agg(lambda x: x.mode()[0])
noun_labels = expanded.groupby(["video_id", "block"])["noun_class"].agg(lambda x: x.mode()[0])

print(f"Relevant blocks: {len(verb_labels)}")
print(f"Unique verb classes: {verb_labels.nunique()}  Unique noun classes: {noun_labels.nunique()}")

Relevant blocks: 142433
Unique verb classes: 59  Unique noun classes: 271


In [44]:
# Load embeddings only for blocks that have a label
all_embeddings = []
all_verb_labels = []
all_noun_labels = []

for pkl_path in sorted(EMBEDDINGS_DIR.glob("*.pkl")):
    video_id = pkl_path.stem

    with pkl_path.open("rb") as f:
        payload = pickle.load(f)

    embeddings = payload["embeddings"] if isinstance(payload, dict) else payload

    for block_idx in range(len(embeddings)):
        key = (video_id, block_idx)
        if key not in verb_labels.index:
            continue  # skip irrelevant blocks
        all_embeddings.append(embeddings[block_idx])
        all_verb_labels.append(verb_labels[key])
        all_noun_labels.append(noun_labels[key])

X = torch.tensor(np.stack(all_embeddings), dtype=torch.float32)

# Encode labels as contiguous integers starting from 0
verb_encoder = LabelEncoder().fit(all_verb_labels)
noun_encoder = LabelEncoder().fit(all_noun_labels)

y_verb = torch.tensor(verb_encoder.transform(all_verb_labels), dtype=torch.long)
y_noun = torch.tensor(noun_encoder.transform(all_noun_labels), dtype=torch.long)

n_verb_classes = len(verb_encoder.classes_)
n_noun_classes = len(noun_encoder.classes_)

print(f"Loaded {len(X)} relevant blocks")
print(f"Verb classes: {n_verb_classes}  Noun classes: {n_noun_classes}")

Loaded 7612 relevant blocks
Verb classes: 44  Noun classes: 150


In [45]:
def make_classifier(input_dim, num_classes, hidden_dims=None, dropout=0.25):
    hidden_dims = hidden_dims or [512, 256, 128]
    layers = []
    prev_dim = input_dim
    for hidden_dim in hidden_dims:
        layers.extend([
            nn.Linear(prev_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        ])
        prev_dim = hidden_dim
    layers.append(nn.Linear(prev_dim, num_classes))
    return nn.Sequential(*layers)

def train_model(model, X_train, y_train, X_val, y_val, device, task_name, config):
    model = model.to(device)
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=config["batch_size"], shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    criterion = nn.CrossEntropyLoss()

    if USE_WANDB and wandb.run is not None:
        wandb.watch(model, criterion=criterion, log="gradients", log_freq=100)

    best_val_acc = 0.0
    for epoch in range(1, config["epochs"] + 1):
        model.train()
        total_loss = 0.0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y_batch)

        train_loss = total_loss / len(y_train)
        val_loss, val_acc, _ = evaluate_model(model, X_val, y_val, device, criterion)
        best_val_acc = max(best_val_acc, val_acc)
        print(f"epoch={epoch:02d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.3f}")

        if USE_WANDB and wandb.run is not None:
            wandb.log({
                "epoch": epoch,
                f"{task_name}/train_loss": train_loss,
                f"{task_name}/val_loss": val_loss,
                f"{task_name}/val_acc": val_acc,
                f"{task_name}/best_val_acc": best_val_acc,
            })
    return model

def evaluate_model(model, X_eval, y_eval, device, criterion=None):
    model.eval()
    with torch.no_grad():
        logits = model(X_eval.to(device)).cpu()
        preds = logits.argmax(dim=-1)
        loss = criterion(logits, y_eval).item() if criterion is not None else None
    acc = (preds == y_eval).float().mean().item()
    return loss, acc, preds


def start_wandb_run(task_name, num_classes, num_samples, config_type):
    if not USE_WANDB:
        print("W&B disabled: install with `pip install wandb` to log runs.")
        return None
    run_config = config_type | {
        "task": task_name,
        "num_classes": num_classes,
        "num_samples": num_samples,
        "input_dim": input_dim,
        "device": str(device),
    }
    return wandb.init(project=WANDB_PROJECT, name=f"{task_name}-classifier", config=run_config, reinit=True)

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
input_dim = X.shape[1]
print(f"Device: {device}  Input dim: {input_dim}")

Device: mps  Input dim: 1024


In [46]:
# --- Verb classifier ---
# Drop classes with fewer than 2 samples — stratified split requires at least 2 per class
verb_counts = y_verb.bincount()
valid_verb_mask = verb_counts[y_verb] >= 2
X_verb, y_verb_filtered = X[valid_verb_mask], y_verb[valid_verb_mask]
print(f"Dropped {(~valid_verb_mask).sum().item()} blocks with singleton verb classes")

X_train, X_test, y_train, y_test = train_test_split(
    X_verb,
    y_verb_filtered,
    test_size=VERB_CONFIG["test_size"],
    random_state=VERB_CONFIG["random_state"],
    stratify=y_verb_filtered,
)

print("Training verb classifier...")
verb_run = start_wandb_run("verb", n_verb_classes, len(X_verb), VERB_CONFIG)
verb_model = train_model(
    make_classifier(input_dim, n_verb_classes, VERB_CONFIG["hidden_dims"], VERB_CONFIG["dropout"]),
    X_train,
    y_train,
    X_test,
    y_test,
    device,
    task_name="verb",
    config=VERB_CONFIG,
)

print("\nVerb classifier results:")
verb_loss, verb_acc, _ = evaluate_model(verb_model, X_test, y_test, device, nn.CrossEntropyLoss())
print(f"accuracy: {verb_acc:.3f}  ({int(verb_acc * len(y_test))}/{len(y_test)} correct)")
if USE_WANDB and wandb.run is not None:
    wandb.summary["final_loss"] = verb_loss
    wandb.summary["final_acc"] = verb_acc
    wandb.finish()

Dropped 1 blocks with singleton verb classes
Training verb classifier...


wandb: setting up run 09u3xkk7
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /Users/mike/Documents/USI/2nd_semester/computer_vision/computer-vision/video-classifier/wandb/run-20260506_135108-09u3xkk7
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run verb-classifier
wandb: ⭐️ View project at https://wandb.ai/bettim-univers/action-classifier
wandb: 🚀 View run at https://wandb.ai/bettim-univers/action-classifier/runs/09u3xkk7


epoch=01  train_loss=2.3746  val_loss=1.8570  val_acc=0.472
epoch=02  train_loss=1.7269  val_loss=1.5288  val_acc=0.542
epoch=03  train_loss=1.4421  val_loss=1.3542  val_acc=0.590
epoch=04  train_loss=1.2443  val_loss=1.2242  val_acc=0.615
epoch=05  train_loss=1.0974  val_loss=1.1426  val_acc=0.645
epoch=06  train_loss=0.9687  val_loss=1.0870  val_acc=0.662
epoch=07  train_loss=0.8583  val_loss=1.0399  val_acc=0.666
epoch=08  train_loss=0.7583  val_loss=1.0033  val_acc=0.672
epoch=09  train_loss=0.6774  val_loss=1.0061  val_acc=0.660
epoch=10  train_loss=0.6068  val_loss=0.9842  val_acc=0.681
epoch=11  train_loss=0.5283  val_loss=0.9953  val_acc=0.674
epoch=12  train_loss=0.4632  val_loss=0.9535  val_acc=0.691
epoch=13  train_loss=0.4158  val_loss=0.9141  val_acc=0.712
epoch=14  train_loss=0.3623  val_loss=0.9978  val_acc=0.697
epoch=15  train_loss=0.3396  val_loss=0.9868  val_acc=0.707
epoch=16  train_loss=0.2944  val_loss=0.9435  val_acc=0.721
epoch=17  train_loss=0.2562  val_loss=0.

wandb: updating run metadata


epoch=29  train_loss=0.0934  val_loss=1.1399  val_acc=0.720
epoch=30  train_loss=0.0825  val_loss=1.1819  val_acc=0.727

Verb classifier results:
accuracy: 0.727  (1107/1523 correct)


wandb: uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json
wandb: uploading history steps 0-29, summary
wandb: 
wandb: Run history:
wandb:             epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
wandb: verb/best_val_acc ▁▃▄▅▆▆▆▆▆▇▇▇██████████████████
wandb:   verb/train_loss █▆▅▅▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      verb/val_acc ▁▃▄▅▆▆▆▆▆▇▆▇█▇▇███████████████
wandb:     verb/val_loss █▆▄▃▃▂▂▂▂▂▂▁▁▂▂▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃
wandb: 
wandb: Run summary:
wandb:             epoch 30
wandb:         final_acc 0.72685
wandb:        final_loss 1.18191
wandb: verb/best_val_acc 0.73014
wandb:   verb/train_loss 0.08253
wandb:      verb/val_acc 0.72685
wandb:     verb/val_loss 1.18191
wandb: 
wandb: 🚀 View run verb-classifier at: https://wandb.ai/bettim-univers/action-classifier/runs/09u3xkk7
wandb: ⭐️ View project at: https://wandb.ai/bettim-univers/action-classifier
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wan

In [47]:
# --- Noun classifier ---
# Drop classes with fewer than 2 samples — stratified split requires at least 2 per class
noun_counts = y_noun.bincount()
valid_noun_mask = noun_counts[y_noun] >= 2
X_noun, y_noun_filtered = X[valid_noun_mask], y_noun[valid_noun_mask]
print(f"Dropped {(~valid_noun_mask).sum().item()} blocks with singleton noun classes")

X_train, X_test, y_train, y_test = train_test_split(
    X_noun,
    y_noun_filtered,
    test_size=NOUN_CONFIG["test_size"],
    random_state=NOUN_CONFIG["random_state"],
    stratify=y_noun_filtered,
)

print("Training noun classifier...")
noun_run = start_wandb_run("noun", n_noun_classes, len(X_noun), NOUN_CONFIG)
noun_model = train_model(
    make_classifier(input_dim, n_noun_classes, NOUN_CONFIG["hidden_dims"], NOUN_CONFIG["dropout"]),
    X_train,
    y_train,
    X_test,
    y_test,
    device,
    task_name="noun",
    config=NOUN_CONFIG,
)

print("\nNoun classifier results:")
noun_loss, noun_acc, _ = evaluate_model(noun_model, X_test, y_test, device, nn.CrossEntropyLoss())
print(f"accuracy: {noun_acc:.3f}  ({int(noun_acc * len(y_test))}/{len(y_test)} correct)")
if USE_WANDB and wandb.run is not None:
    wandb.summary["final_loss"] = noun_loss
    wandb.summary["final_acc"] = noun_acc
    wandb.finish()

Dropped 3 blocks with singleton noun classes
Training noun classifier...


wandb: setting up run l47c5p2v
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /Users/mike/Documents/USI/2nd_semester/computer_vision/computer-vision/video-classifier/wandb/run-20260506_135114-l47c5p2v
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run noun-classifier
wandb: ⭐️ View project at https://wandb.ai/bettim-univers/action-classifier
wandb: 🚀 View run at https://wandb.ai/bettim-univers/action-classifier/runs/l47c5p2v


epoch=01  train_loss=4.0314  val_loss=3.2141  val_acc=0.302
epoch=02  train_loss=3.0547  val_loss=2.6205  val_acc=0.399
epoch=03  train_loss=2.5426  val_loss=2.2785  val_acc=0.449
epoch=04  train_loss=2.1737  val_loss=2.0542  val_acc=0.495
epoch=05  train_loss=1.8977  val_loss=1.8930  val_acc=0.538
epoch=06  train_loss=1.6785  val_loss=1.7575  val_acc=0.570
epoch=07  train_loss=1.4904  val_loss=1.6539  val_acc=0.591
epoch=08  train_loss=1.3107  val_loss=1.5777  val_acc=0.605
epoch=09  train_loss=1.1841  val_loss=1.5219  val_acc=0.619
epoch=10  train_loss=1.0545  val_loss=1.4581  val_acc=0.631
epoch=11  train_loss=0.9364  val_loss=1.4011  val_acc=0.639
epoch=12  train_loss=0.8422  val_loss=1.3635  val_acc=0.647
epoch=13  train_loss=0.7536  val_loss=1.3231  val_acc=0.654
epoch=14  train_loss=0.6677  val_loss=1.3215  val_acc=0.662
epoch=15  train_loss=0.5982  val_loss=1.3115  val_acc=0.666
epoch=16  train_loss=0.5494  val_loss=1.2661  val_acc=0.673
epoch=17  train_loss=0.4776  val_loss=1.

wandb: updating run metadata
wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml
wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json
wandb: uploading history steps 0-29, summary
wandb: 
wandb: Run history:
wandb:             epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
wandb: noun/best_val_acc ▁▃▄▄▅▆▆▆▇▇▇▇▇▇▇███████████████
wandb:   noun/train_loss █▆▅▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      noun/val_acc ▁▃▄▄▅▆▆▆▇▇▇▇▇▇▇███████████████
wandb:     noun/val_loss █▆▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:             epoch 30
wandb:         final_acc 0.69842
wandb:        final_loss 1.2904
wandb: noun/best_val_acc 0.70105
wandb:   noun/train_loss 0.14764
wandb:      noun/val_acc 0.69842
wandb:     noun/val_loss 1.2904
wandb: 
wandb: 🚀 View run noun-classifier at: https://wandb.ai/bettim-univers/action-classifier/runs/l47c5p2v
wandb: ⭐️ View project at: https://wandb.ai